<a href="https://colab.research.google.com/github/EngrOluchi/Aufgaben/blob/main/Digital_Twin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# STAGE 1 — CELL 1
# Reference configuration and traceable assumptions.
# No PINN training or dynamic simulation is performed here.

import math
import numpy as np
import pandas as pd
from IPython.display import display

# ------------------------------------------------------------
# 1. DOCUMENTED VEHICLE REFERENCE
# Source: supplied Schurr thesis, vehicle description, pp. 10–16.
# These are not measurements of your finished electric conversion.
# ------------------------------------------------------------

vehicle_geometry = {
    "length_m": 2.820,
    "width_m": 1.370,
    "height_m": 1.780,
    "wheelbase_m": 1.920,
    "rear_track_m": 1.140,
    "number_of_wheels": 3,
}

# New dictionary: vehicle_reference separates original-vehicle
# specifications from conversion assumptions.
vehicle_reference = {
    "original_ready_mass_kg": 395.0,
    "original_mass_with_driver_kg": 470.0,
    "gross_mass_limit_kg": 645.0,
    "original_max_speed_kmh": 61.0,
}

# ------------------------------------------------------------
# 2. MOTOR REFERENCE
# Selected thesis variant: Heinzmann PMS 120, 4500 rpm.
# Source: thesis p. 75.
# Maximum ratings are not treated as continuous ratings.
# ------------------------------------------------------------

motor_specs = {
    "manufacturer": "Heinzmann",
    "model": "PMS 120",
    "rated_power_W": 6000.0,
    "nominal_torque_Nm": 12.7,
    "maximum_torque_Nm": 45.0,
    "maximum_speed_rpm": 4500.0,
    "maximum_power_W": 16900.0,
    "overload_duration_s": None,
    "rating_conventions_verified": False,
}

# Do not infer nominal speed from the listed maximum speed.
# Detailed motor operating limits will be established later.

# ------------------------------------------------------------
# 3. THESIS SIMULATION INPUTS
# New dictionary: thesis_model retains the reproduction baseline.
# Source: thesis pp. 51–52 and final concept pp. 72–74.
# ------------------------------------------------------------

thesis_model = {
    "total_mass_kg": 645.0,
    "radius_m": 0.205,
    "wheel_mass_kg": 12.0,          # Thesis assumption
    "frontal_area_m2": 1.60,        # Thesis approximation
    "drag_coefficient": 0.60,       # Thesis assumption
    "Crr": 0.015,
    "rho": 1.184,
    "overall_efficiency": 0.92,     # Lumped thesis assumption
    "reduction": 6.1,               # Final proposed configuration
    "voltage_V": 48.0,
}

# ------------------------------------------------------------
# 4. PROVISIONAL CONVERSION MASS BUDGET
# New dictionary: mass_budget documents how conversion mass
# is calculated. All entries below are our assumptions.
# ------------------------------------------------------------

mass_budget = {
    "removed_mass_kg": 60.0,
    "battery_mass_kg": 60.0,
    "motor_mass_kg": 15.0,
    "inverter_mass_kg": 5.0,
    "other_added_mass_kg": 15.0,
    "driver_mass_kg": 75.0,
    "additional_payload_kg": 140.0,
}

# ------------------------------------------------------------
# 5. ACTIVE MODEL CONFIGURATION
# The established 'assumed' name is retained.
# Provenance is recorded separately below.
# ------------------------------------------------------------

assumed = {
    "vehicle_mass_kg": (
        vehicle_reference["original_ready_mass_kg"]
        - mass_budget["removed_mass_kg"]
        + mass_budget["battery_mass_kg"]
        + mass_budget["motor_mass_kg"]
        + mass_budget["inverter_mass_kg"]
        + mass_budget["other_added_mass_kg"]
    ),
    # payload_kg includes driver plus additional passengers/cargo.
    "payload_kg": (
        mass_budget["driver_mass_kg"]
        + mass_budget["additional_payload_kg"]
    ),
    "radius_m": thesis_model["radius_m"],
    "reduction": thesis_model["reduction"],
    "frontal_area_m2": thesis_model["frontal_area_m2"],
    "drag_coefficient": thesis_model["drag_coefficient"],
    "Crr": thesis_model["Crr"],
    "rho": thesis_model["rho"],
    "voltage_V": thesis_model["voltage_V"],
    "capacity_Ah": 150.0,          # Our provisional assumption
    "eta_gear": 0.95,              # Our provisional assumption
    "initial_soc": 0.80,
    "minimum_soc": 0.10,
    "maximum_soc": 0.95,
    "auxiliary_W": 150.0,
    "cg_x_m": 0.70,
    "cg_height_m": 0.65,
    "mu": 0.70,
}

assumed["CdA_m2"] = (
    assumed["frontal_area_m2"] * assumed["drag_coefficient"]
)

# New configuration switch:
# reproduce the thesis first; detailed component-loss modelling later.
model_settings = {
    "loss_model": "thesis_lumped",
    "pinn_enabled": False,
    "physical_validation_status": "not_validated",
}

environment = {
    "gravity_mps2": 9.81,
    "ambient_temperature_C": 25.0,
    "road_grade_rad": 0.0,
    "wind_speed_mps": 0.0,
}

architecture = {
    "propulsion": "electric conversion reference",
    "steering": "single front wheel",
    "driven_wheels": "two rear wheels",
    "motor": "Heinzmann PMS 120, 4500 rpm variant",
    "traction_power_path": "battery directly connected to inverter",
    "transmission": "retained gearbox fixed in fourth gear",
    "differential": "retained rear differential",
    "traction_battery": "48 V nominal; detailed selection provisional",
}

# ------------------------------------------------------------
# 6. DERIVED QUANTITIES
# ------------------------------------------------------------

derived = {}

derived["total_mass_kg"] = (
    assumed["vehicle_mass_kg"] + assumed["payload_kg"]
)

derived["wheel_diameter_m"] = 2.0 * assumed["radius_m"]

derived["wheel_inertia_kgm2"] = (
    0.5
    * thesis_model["wheel_mass_kg"]
    * assumed["radius_m"]**2
)

derived["equivalent_mass_kg"] = (
    derived["total_mass_kg"]
    + vehicle_geometry["number_of_wheels"]
    * derived["wheel_inertia_kgm2"]
    / assumed["radius_m"]**2
)

derived["nominal_battery_energy_kWh"] = (
    assumed["voltage_V"] * assumed["capacity_Ah"] / 1000.0
)

# ------------------------------------------------------------
# 7. NUMERICAL AND CONFIGURATION CHECKS
# ------------------------------------------------------------

assert all(
    value > 0 for value in vehicle_geometry.values()
)

assert assumed["vehicle_mass_kg"] > 0
assert assumed["payload_kg"] >= 0
assert assumed["radius_m"] > 0
assert assumed["reduction"] > 0
assert 0 < assumed["eta_gear"] <= 1
assert 0 < thesis_model["overall_efficiency"] <= 1

assert (
    0 <= assumed["minimum_soc"]
    < assumed["initial_soc"]
    <= assumed["maximum_soc"] <= 1
)

assert 0 < assumed["cg_x_m"] < vehicle_geometry["wheelbase_m"]

assert (
    derived["total_mass_kg"]
    <= vehicle_reference["gross_mass_limit_kg"] + 1e-9
), "The selected loading exceeds the documented gross-mass limit."

assert math.isclose(
    derived["equivalent_mass_kg"]
    - derived["total_mass_kg"],
    0.5
    * vehicle_geometry["number_of_wheels"]
    * thesis_model["wheel_mass_kg"],
    abs_tol=1e-10,
)

# ------------------------------------------------------------
# 8. TRACEABLE PARAMETER SUMMARY
# ------------------------------------------------------------

parameter_register = pd.DataFrame([
    (
        "Wheelbase",
        vehicle_geometry["wheelbase_m"], "m",
        "Documented vehicle reference", "Thesis p. 10"
    ),
    (
        "Rear track",
        vehicle_geometry["rear_track_m"], "m",
        "Documented vehicle reference", "Thesis p. 10"
    ),
    (
        "Gross-mass limit",
        vehicle_reference["gross_mass_limit_kg"], "kg",
        "Documented vehicle reference", "Thesis p. 11"
    ),
    (
        "Converted ready-for-use mass",
        assumed["vehicle_mass_kg"], "kg",
        "Calculated from our assumed conversion budget",
        "Project assumption"
    ),
    (
        "Driver plus additional payload",
        assumed["payload_kg"], "kg",
        "Our selected loading", "Project assumption"
    ),
    (
        "Total moving mass",
        derived["total_mass_kg"], "kg",
        "Calculated loading; matches thesis full-load case",
        "Derived"
    ),
    (
        "Wheel radius",
        assumed["radius_m"], "m",
        "Thesis model input; loaded radius unverified",
        "Thesis p. 52"
    ),
    (
        "Equivalent acceleration mass",
        derived["equivalent_mass_kg"], "kg",
        "Calculated using assumed solid-cylinder wheels",
        "Derived"
    ),
    (
        "Frontal area",
        assumed["frontal_area_m2"], "m²",
        "Thesis approximation", "Thesis pp. 51–52"
    ),
    (
        "Drag coefficient",
        assumed["drag_coefficient"], "—",
        "Thesis assumption", "Thesis p. 52"
    ),
    (
        "Nominal battery capacity",
        assumed["capacity_Ah"], "Ah",
        "Our provisional assumption", "Project assumption"
    ),
    (
        "Nominal battery energy",
        derived["nominal_battery_energy_kWh"], "kWh",
        "Calculated; capacity remains assumed", "Derived"
    ),
], columns=["Parameter", "Value", "Unit", "Evidence", "Source"])

display(parameter_register)

print("\nPASS: baseline configuration checks completed.")
print("Current scope: Stage 1 configuration only.")
print("PINN training is deferred.")
print("No physical validation is claimed.")
print(
    "Loss accounting: use the thesis overall efficiency OR detailed "
    "component losses later; never apply both together."
)

,Parameter,Value,Unit,Evidence,Source
0,Wheelbase,1.920,m,Documented vehicle reference,Thesis p. 10
1,Rear track,1.140,m,Documented vehicle reference,Thesis p. 10
2,Gross-mass limit,645.000,kg,Documented vehicle reference,Thesis p. 11
3,Converted ready-for-use mass,430.000,kg,Calculated from our assumed conversion budget,Project assumption
4,Driver plus additional payload,215.000,kg,Our selected loading,Project assumption
5,Total moving mass,645.000,kg,Calculated loading; matches thesis full-load case,Derived
6,Wheel radius,0.205,m,Thesis model input; loaded radius unverified,Thesis p. 52
7,Equivalent acceleration mass,663.000,kg,Calculated using assumed solid-cylinder wheels,Derived
8,Frontal area,1.600,m²,Thesis approximation,Thesis pp. 51–52
9,Drag coefficient,0.600,—,Thesis assumption,Thesis p. 52



PASS: baseline configuration checks completed.
Current scope: Stage 1 configuration only.
PINN training is deferred.
No physical validation is claimed.
Loss accounting: use the thesis overall efficiency OR detailed component losses later; never apply both together.


In [2]:
# STAGE 1 — CELL 2
# Subsystem interfaces and interactive 3D packaging layout.
# No dynamic simulation or PINN training is performed.

required_objects = [
    "vehicle_geometry", "vehicle_reference", "motor_specs",
    "assumed", "derived", "architecture", "model_settings",
    "np", "pd", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted Stage 1 Cell 1 first. Missing: "
        + ", ".join(missing_objects)
    )

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import clear_output

try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass

# ------------------------------------------------------------
# 1. SUBSYSTEM INTERFACES
# ------------------------------------------------------------

interface_table = pd.DataFrame([
    (
        "Battery / BMS",
        "Current demand; ambient temperature",
        "DC voltage; SOC; temperature; limits"
    ),
    (
        "Inverter",
        "DC voltage; motor state; torque command",
        "Motor excitation; battery current; losses"
    ),
    (
        "Motor",
        "Electrical excitation; mechanical load",
        "Shaft torque; shaft speed; losses"
    ),
    (
        "Gearbox / differential",
        "Motor torque and speed",
        "Rear-wheel torque and speed"
    ),
    (
        "Vehicle dynamics",
        "Wheel forces; brakes; gradient; wind",
        "Speed; acceleration; wheel loads"
    ),
    (
        "Controller",
        "Driver commands; states; protection limits",
        "Permitted drive and braking commands"
    ),
], columns=["Subsystem", "Inputs", "Outputs"])

display(interface_table)

# ------------------------------------------------------------
# 2. GEOMETRY ASSUMPTIONS
# New dictionary: layout_assumptions contains display geometry,
# not physical-model calibration parameters.
# ------------------------------------------------------------

layout_assumptions = {
    "rear_overhang_m": 0.45,
    "tyre_width_m": 0.12,
    "chassis_width_m": 0.85,
    "chassis_centre_z_m": 0.29,
    "chassis_thickness_m": 0.08,
}

# Component envelopes: all sizes and mounting positions ASSUMED.
# centre = [x, y, z]; size = [length, width, height], in metres.
# The motor and transmission are separated along y to illustrate
# an axle-parallel mechanical arrangement.
packaging = {
    "Battery": {
        "centre": [0.72, 0.0, 0.48],
        "size": [0.75, 0.65, 0.24],
        "colour": "#27ae60",
    },
    "Inverter": {
        "centre": [0.12, 0.32, 0.72],
        "size": [0.30, 0.22, 0.12],
        "colour": "#f39c12",
    },
    "Motor": {
        "centre": [0.0, -0.30, 0.43],
        "size": [0.24, 0.28, 0.24],
        "colour": "#e74c3c",
    },
    "Gearbox / differential": {
        "centre": [0.0, 0.06, 0.35],
        "size": [0.30, 0.30, 0.22],
        "colour": "#9b59b6",
    },
}

D = vehicle_geometry["wheelbase_m"]
E = vehicle_geometry["rear_track_m"]
L = vehicle_geometry["length_m"]
W = vehicle_geometry["width_m"]
H = vehicle_geometry["height_m"]
r = assumed["radius_m"]

body_rear_x_m = -layout_assumptions["rear_overhang_m"]
body_front_x_m = body_rear_x_m + L

wheel_centres = {
    "Front wheel": np.array([D, 0.0, r]),
    "Rear left wheel": np.array([0.0, E / 2.0, r]),
    "Rear right wheel": np.array([0.0, -E / 2.0, r]),
}

# Geometry checks.
assert np.isclose(body_front_x_m - body_rear_x_m, L)
assert body_front_x_m > D

assert np.isclose(
    wheel_centres["Front wheel"][0]
    - wheel_centres["Rear left wheel"][0],
    D
)

assert np.isclose(
    wheel_centres["Rear left wheel"][1]
    - wheel_centres["Rear right wheel"][1],
    E
)

for centre in wheel_centres.values():
    assert np.isclose(centre[2] - r, 0.0)

assert E + layout_assumptions["tyre_width_m"] <= W

for unit in packaging.values():
    centre = np.array(unit["centre"])
    size = np.array(unit["size"])
    assert np.all(size > 0)
    assert centre[2] - size[2] / 2.0 > 0

# ------------------------------------------------------------
# 3. DRAWING FUNCTIONS
# ------------------------------------------------------------

def add_component_box(
    fig, name, centre, size, colour, opacity=0.8
):
    """Draw an envelope, not detailed component CAD."""
    vertices = np.array([
        [-1, -1, -1], [1, -1, -1],
        [1, 1, -1], [-1, 1, -1],
        [-1, -1, 1], [1, -1, 1],
        [1, 1, 1], [-1, 1, 1],
    ], dtype=float)

    vertices = (
        vertices * np.asarray(size) / 2.0
        + np.asarray(centre)
    )

    triangles = np.array([
        [0, 1, 2], [0, 2, 3],
        [4, 5, 6], [4, 6, 7],
        [0, 1, 5], [0, 5, 4],
        [1, 2, 6], [1, 6, 5],
        [2, 3, 7], [2, 7, 6],
        [3, 0, 4], [3, 4, 7],
    ])

    fig.add_trace(go.Mesh3d(
        x=vertices[:, 0],
        y=vertices[:, 1],
        z=vertices[:, 2],
        i=triangles[:, 0],
        j=triangles[:, 1],
        k=triangles[:, 2],
        color=colour,
        opacity=opacity,
        name=name,
        showlegend=True,
        hovertemplate=(
            name + "<br>Assumed geometry"
            "<br>x=%{x:.2f} m"
            "<br>y=%{y:.2f} m"
            "<br>z=%{z:.2f} m<extra></extra>"
        ),
    ))

def build_layout(
    selected_unit="All units",
    show_body=True,
    show_labels=True,
    view="Isometric"
):
    fig = go.Figure()
    angle = np.linspace(0.0, 2.0 * np.pi, 100)

    # Wheels: sidewall outlines using an assumed tyre width.
    for name, centre in wheel_centres.items():
        x0, y0, z0 = centre

        for index, side in enumerate([-0.5, 0.5]):
            fig.add_trace(go.Scatter3d(
                x=x0 + r * np.cos(angle),
                y=np.full_like(
                    angle,
                    y0 + side * layout_assumptions["tyre_width_m"]
                ),
                z=z0 + r * np.sin(angle),
                mode="lines",
                line=dict(color="#222222", width=6),
                name=name,
                legendgroup=name,
                showlegend=(index == 0),
            ))

    # Rear axle location.
    fig.add_trace(go.Scatter3d(
        x=[0.0, 0.0],
        y=[-E / 2.0, E / 2.0],
        z=[r, r],
        mode="lines",
        line=dict(color="#7f8c8d", width=7),
        name="Rear axle — schematic",
    ))

    add_component_box(
        fig,
        "Chassis envelope",
        [
            (body_rear_x_m + body_front_x_m) / 2.0,
            0.0,
            layout_assumptions["chassis_centre_z_m"]
        ],
        [
            L,
            layout_assumptions["chassis_width_m"],
            layout_assumptions["chassis_thickness_m"]
        ],
        "#607d8b",
        opacity=0.25,
    )

    for name, unit in packaging.items():
        opacity = (
            0.90
            if selected_unit in ["All units", name]
            else 0.10
        )

        add_component_box(
            fig, name,
            unit["centre"], unit["size"],
            unit["colour"], opacity
        )

        if show_labels:
            fig.add_trace(go.Scatter3d(
                x=[unit["centre"][0]],
                y=[unit["centre"][1]],
                z=[
                    unit["centre"][2]
                    + unit["size"][2] / 2.0
                    + 0.08
                ],
                mode="text",
                text=[name],
                textfont=dict(size=11, color="#172b4d"),
                showlegend=False,
                hoverinfo="skip",
            ))

    # Illustrative steering column.
    fig.add_trace(go.Scatter3d(
        x=[D, D - 0.25],
        y=[0.0, 0.0],
        z=[r + 0.08, 1.15],
        mode="lines",
        line=dict(color="#00a6c8", width=7),
        name="Steering — assumed geometry",
    ))

    if show_body:
        add_component_box(
            fig,
            "Body envelope",
            [
                (body_rear_x_m + body_front_x_m) / 2.0,
                0.0, 0.95
            ],
            [L, W, 1.10],
            "#3498db",
            opacity=0.07,
        )

        add_component_box(
            fig,
            "Canopy envelope",
            [0.65, 0.0, H - 0.025],
            [2.05, W, 0.05],
            "#d9c9a3",
            opacity=0.30,
        )

    cameras = {
        "Isometric": dict(
            eye=dict(x=1.6, y=1.6, z=0.9),
            up=dict(x=0, y=0, z=1)
        ),
        "Side": dict(
            eye=dict(x=0, y=2.5, z=0),
            up=dict(x=0, y=0, z=1),
            projection=dict(type="orthographic")
        ),
        "Front": dict(
            eye=dict(x=2.5, y=0, z=0),
            up=dict(x=0, y=0, z=1),
            projection=dict(type="orthographic")
        ),
        "Top": dict(
            eye=dict(x=0, y=0, z=2.5),
            up=dict(x=1, y=0, z=0),
            projection=dict(type="orthographic")
        ),
    }

    fig.update_layout(
        title=(
            "Stage 1 | Electric tricycle functional-unit layout"
            "<br><sup>Reference overall dimensions; "
            "assumed component positions and body shape</sup>"
        ),
        scene=dict(
            xaxis_title="Forward x [m]",
            yaxis_title="Left y [m]",
            zaxis_title="Height z [m]",
            aspectmode="data",
            camera=cameras[view],
            zaxis=dict(range=[0.0, H + 0.15]),
        ),
        height=650,
        margin=dict(l=0, r=0, t=80, b=0),
    )

    return fig

# ------------------------------------------------------------
# 4. INTERACTIVE CONTROLS
# ------------------------------------------------------------

unit_selector = widgets.Dropdown(
    options=["All units"] + list(packaging),
    value="All units",
    description="Highlight:",
)

view_selector = widgets.Dropdown(
    options=["Isometric", "Side", "Front", "Top"],
    value="Isometric",
    description="View:",
)

body_toggle = widgets.Checkbox(
    value=True, description="Body / canopy", indent=False
)

label_toggle = widgets.Checkbox(
    value=True, description="Labels", indent=False
)

layout_output = widgets.Output()

def refresh_layout(change=None):
    with layout_output:
        clear_output(wait=True)
        fig = build_layout(
            selected_unit=unit_selector.value,
            show_body=body_toggle.value,
            show_labels=label_toggle.value,
            view=view_selector.value,
        )
        fig.show(config={"displaylogo": False})

for control in [
    unit_selector, view_selector, body_toggle, label_toggle
]:
    control.observe(refresh_layout, names="value")

display(widgets.HBox(
    [unit_selector, view_selector, body_toggle, label_toggle],
    layout=widgets.Layout(flex_flow="row wrap")
))

display(layout_output)
refresh_layout()

print("PASS: wheelbase, rear track and ground-contact checks.")
print("PASS: assumed component envelopes lie above ground.")
print("No full clearance, structural or dynamic validation performed.")

,Subsystem,Inputs,Outputs
0,Battery / BMS,Current demand; ambient temperature,DC voltage; SOC; temperature; limits
1,Inverter,DC voltage; motor state; torque command,Motor excitation; battery current; losses
2,Motor,Electrical excitation; mechanical load,Shaft torque; shaft speed; losses
3,Gearbox / differential,Motor torque and speed,Rear-wheel torque and speed
4,Vehicle dynamics,Wheel forces; brakes; gradient; wind,Speed; acceleration; wheel loads
5,Controller,Driver commands; states; protection limits,Permitted drive and braking commands


Output()

PASS: wheelbase, rear track and ground-contact checks.
PASS: assumed component envelopes lie above ground.
No full clearance, structural or dynamic validation performed.


In [3]:
# STAGE 1 — CELL 3
# Static axle loads and steady road-load checks.
# No acceleration, battery dynamics or PINN training.

required_objects = [
    "vehicle_geometry", "vehicle_reference", "motor_specs",
    "thesis_model", "assumed", "derived", "environment",
    "model_settings", "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted Stage 1 Cells 1 and 2 first. Missing: "
        + ", ".join(missing_objects)
    )

if model_settings["loss_model"] != "thesis_lumped":
    raise RuntimeError(
        "This cell uses thesis_lumped loss accounting. "
        "Check the selected loss model before continuing."
    )

# ------------------------------------------------------------
# 1. RECALCULATE FROM CURRENT CONFIGURATION
# ------------------------------------------------------------

derived["total_mass_kg"] = (
    assumed["vehicle_mass_kg"] + assumed["payload_kg"]
)

assumed["CdA_m2"] = (
    assumed["frontal_area_m2"] * assumed["drag_coefficient"]
)

m = derived["total_mass_kg"]
g = environment["gravity_mps2"]
D = vehicle_geometry["wheelbase_m"]
x_cg = assumed["cg_x_m"]

assert m > 0
assert g > 0
assert 0 < x_cg < D
assert assumed["radius_m"] > 0
assert assumed["reduction"] > 0
assert assumed["Crr"] >= 0
assert assumed["rho"] > 0
assert assumed["CdA_m2"] >= 0
assert 0 < thesis_model["overall_efficiency"] <= 1

assert (
    m <= vehicle_reference["gross_mass_limit_kg"] + 1e-9
), "The selected loading exceeds the documented gross-mass limit."

# This particular check is explicitly level-road, still-air only.
assert np.isclose(environment["road_grade_rad"], 0.0), (
    "This cell assumes a level road. Restore road_grade_rad to 0 "
    "or wait for the gradient model."
)

assert np.isclose(environment["wind_speed_mps"], 0.0), (
    "This cell assumes zero wind."
)

# ------------------------------------------------------------
# 2. STATIC WHEEL LOADS
# ------------------------------------------------------------

derived["front_static_load_N"] = m * g * x_cg / D
derived["rear_static_load_N"] = (
    m * g - derived["front_static_load_N"]
)

derived["rear_each_static_load_N"] = (
    derived["rear_static_load_N"] / 2.0
)

assert np.isclose(
    derived["front_static_load_N"]
    + derived["rear_static_load_N"],
    m * g,
)

assert derived["front_static_load_N"] > 0
assert derived["rear_each_static_load_N"] > 0

wheel_load_table = pd.DataFrame([
    (
        "Front wheel",
        derived["front_static_load_N"],
        100.0 * derived["front_static_load_N"] / (m * g),
    ),
    (
        "Rear left wheel",
        derived["rear_each_static_load_N"],
        100.0 * derived["rear_each_static_load_N"] / (m * g),
    ),
    (
        "Rear right wheel",
        derived["rear_each_static_load_N"],
        100.0 * derived["rear_each_static_load_N"] / (m * g),
    ),
], columns=["Wheel", "Static load [N]", "Share of weight [%]"])

display(wheel_load_table.round(3))

# ------------------------------------------------------------
# 3. STEADY ROAD-LOAD FUNCTION
# ------------------------------------------------------------

def calculate_steady_road_load(speed_kmh):
    """
    Forward steady travel, level road and zero wind.

    At zero speed, rolling force is set to zero for reporting.
    The simple moving rolling-resistance model is not a
    standstill or breakaway-friction model.
    """
    speed_kmh = np.atleast_1d(
        np.asarray(speed_kmh, dtype=float)
    )

    if (
        not np.all(np.isfinite(speed_kmh))
        or np.any(speed_kmh < 0)
    ):
        raise ValueError("Speeds must be finite and nonnegative.")

    speed_mps = speed_kmh / 3.6

    total_mass_kg = (
        assumed["vehicle_mass_kg"] + assumed["payload_kg"]
    )

    rolling_force_N = np.where(
        speed_mps > 0,
        assumed["Crr"]
        * total_mass_kg
        * environment["gravity_mps2"],
        0.0,
    )

    aero_force_N = (
        0.5
        * assumed["rho"]
        * assumed["CdA_m2"]
        * speed_mps**2
    )

    total_resistance_N = rolling_force_N + aero_force_N
    wheel_power_W = total_resistance_N * speed_mps

    upstream_power_W = (
        wheel_power_W / thesis_model["overall_efficiency"]
    )

    motor_rpm = (
        speed_mps
        / assumed["radius_m"]
        * assumed["reduction"]
        * 60.0
        / (2.0 * np.pi)
    )

    return pd.DataFrame({
        "speed_kmh": speed_kmh,
        "rolling_force_N": rolling_force_N,
        "aero_force_N": aero_force_N,
        "total_resistance_N": total_resistance_N,
        "wheel_power_W": wheel_power_W,
        "upstream_traction_power_estimate_W": upstream_power_W,
        "motor_rpm": motor_rpm,
        "within_motor_speed_reference": (
            motor_rpm <= motor_specs["maximum_speed_rpm"]
        ),
    })

road_load_result = calculate_steady_road_load(
    np.linspace(0.0, 60.0, 121)
)

road_load_check_table = calculate_steady_road_load(
    [0.0, 10.0, 30.0, 50.0, 60.0]
)

display(road_load_check_table.round(3))

# ------------------------------------------------------------
# 4. NUMERICAL CHECKS
# ------------------------------------------------------------

zero_speed = calculate_steady_road_load([0.0])
assert np.isclose(zero_speed["wheel_power_W"].iloc[0], 0.0)

speed_scaling_check = calculate_steady_road_load([20.0, 40.0])

assert np.isclose(
    speed_scaling_check["aero_force_N"].iloc[1],
    4.0 * speed_scaling_check["aero_force_N"].iloc[0],
)

assert np.all(
    road_load_result["upstream_traction_power_estimate_W"]
    >= road_load_result["wheel_power_W"] - 1e-9
)

# Kinematic reference only: not a predicted achievable top speed.
derived["speed_at_max_motor_rpm_kmh"] = (
    motor_specs["maximum_speed_rpm"]
    * 2.0 * np.pi / 60.0
    * assumed["radius_m"]
    / assumed["reduction"]
    * 3.6
)

# ------------------------------------------------------------
# 5. PLOTS
# ------------------------------------------------------------

resistance_figure = go.Figure()

for column, label in [
    ("rolling_force_N", "Rolling resistance"),
    ("aero_force_N", "Aerodynamic resistance"),
    ("total_resistance_N", "Total resistance"),
]:
    resistance_figure.add_trace(go.Scatter(
        x=road_load_result["speed_kmh"],
        y=road_load_result[column],
        mode="lines",
        name=label,
    ))

resistance_figure.update_layout(
    title="Baseline road resistance | level road, zero wind",
    xaxis_title="Vehicle speed [km/h]",
    yaxis_title="Force [N]",
    template="plotly_white",
    height=400,
)

resistance_figure.show()

power_figure = go.Figure()

power_figure.add_trace(go.Scatter(
    x=road_load_result["speed_kmh"],
    y=road_load_result["wheel_power_W"] / 1000.0,
    name="Wheel power",
))

power_figure.add_trace(go.Scatter(
    x=road_load_result["speed_kmh"],
    y=(
        road_load_result["upstream_traction_power_estimate_W"]
        / 1000.0
    ),
    name="Lumped upstream traction-power estimate",
))

power_figure.add_vline(
    x=derived["speed_at_max_motor_rpm_kmh"],
    line_dash="dash",
    line_color="red",
)

power_figure.update_layout(
    title=(
        "Steady traction power requirement"
        "<br><sup>Red line: speed corresponding to "
        "4500 motor rpm; no feasibility claim</sup>"
    ),
    xaxis_title="Vehicle speed [km/h]",
    yaxis_title="Power [kW]",
    template="plotly_white",
    height=420,
)

power_figure.show()

print("PASS: static wheel loads sum to vehicle weight.")
print("PASS: zero-speed wheel power is zero.")
print("PASS: aerodynamic force follows speed squared.")
print("PASS: lumped loss accounting is consistent.")

print(
    "Speed corresponding to maximum motor RPM: "
    f"{derived['speed_at_max_motor_rpm_kmh']:.2f} km/h"
)

print(
    "Power curves show requirements, not proof that the "
    "motor/battery can deliver them."
)

,Wheel,Static load [N],Share of weight [%]
0,Front wheel,2306.883,36.458
1,Rear left wheel,2010.284,31.771
2,Rear right wheel,2010.284,31.771


,speed_kmh,rolling_force_N,aero_force_N,total_resistance_N,wheel_power_W,upstream_traction_power_estimate_W,motor_rpm,within_motor_speed_reference
0,0.0,0.000,0.000,0.000,0.000,0.000,0.000,True
1,10.0,94.912,4.385,99.297,275.825,299.810,789.305,True
2,30.0,94.912,39.467,134.378,1119.820,1217.196,2367.915,True
3,50.0,94.912,109.630,204.541,2840.852,3087.883,3946.525,True
4,60.0,94.912,157.867,252.778,4212.974,4579.319,4735.830,False


PASS: static wheel loads sum to vehicle weight.
PASS: zero-speed wheel power is zero.
PASS: aerodynamic force follows speed squared.
PASS: lumped loss accounting is consistent.
Speed corresponding to maximum motor RPM: 57.01 km/h
Power curves show requirements, not proof that the motor/battery can deliver them.


In [4]:
# STAGE 1 — CELL 4
# Final baseline checks, summary and configuration checkpoint.
# No dynamic simulation or PINN training.

import json
from copy import deepcopy

required_objects = [
    "vehicle_geometry", "vehicle_reference", "motor_specs",
    "thesis_model", "mass_budget", "assumed", "derived",
    "environment", "architecture", "model_settings",
    "packaging", "layout_assumptions", "parameter_register",
    "calculate_steady_road_load", "np", "pd", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted Stage 1 Cells 1–3 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# 1. CHECK CURRENT CONFIGURATION
# ------------------------------------------------------------

current_total_mass_kg = (
    assumed["vehicle_mass_kg"] + assumed["payload_kg"]
)

current_weight_N = (
    current_total_mass_kg * environment["gravity_mps2"]
)

current_front_load_N = (
    current_weight_N
    * assumed["cg_x_m"]
    / vehicle_geometry["wheelbase_m"]
)

current_rear_load_N = (
    current_weight_N - current_front_load_N
)

baseline_checks = [
    (
        "Loading within gross-mass limit",
        0 < current_total_mass_kg
        <= vehicle_reference["gross_mass_limit_kg"] + 1e-9
    ),
    (
        "CG between front and rear axles",
        0 < assumed["cg_x_m"]
        < vehicle_geometry["wheelbase_m"]
    ),
    (
        "Positive static axle loads",
        current_front_load_N > 0 and current_rear_load_N > 0
    ),
    (
        "Static axle-load balance",
        np.isclose(
            current_front_load_N + current_rear_load_N,
            current_weight_N
        )
    ),
    (
        "Valid SOC operating window",
        0 <= assumed["minimum_soc"]
        < assumed["initial_soc"]
        <= assumed["maximum_soc"] <= 1
    ),
    (
        "Aerodynamic inputs consistent",
        np.isclose(
            assumed["CdA_m2"],
            assumed["frontal_area_m2"]
            * assumed["drag_coefficient"]
        )
    ),
    (
        "Thesis lumped-loss mode selected",
        model_settings["loss_model"] == "thesis_lumped"
    ),
    (
        "PINN work deferred",
        model_settings["pinn_enabled"] is False
    ),
    (
        "Previous calculations match current mass",
        np.isclose(
            derived["total_mass_kg"],
            current_total_mass_kg
        )
    ),
    (
        "Previous wheel-load calculation is current",
        np.isclose(
            derived["front_static_load_N"],
            current_front_load_N
        )
    ),
]

baseline_check_table = pd.DataFrame(
    [
        {
            "Check": name,
            "Result": "PASS" if bool(passed) else "FAIL"
        }
        for name, passed in baseline_checks
    ]
)

display(baseline_check_table)

if not all(bool(passed) for _, passed in baseline_checks):
    raise RuntimeError(
        "Baseline review failed. Inspect the FAIL rows. "
        "If inputs changed, rerun the affected earlier cells "
        "before creating the checkpoint."
    )

# ------------------------------------------------------------
# 2. BASELINE PERFORMANCE SUMMARY
# ------------------------------------------------------------

baseline_30kmh = calculate_steady_road_load([30.0]).iloc[0]

baseline_summary = pd.DataFrame([
    (
        "Total physical mass",
        current_total_mass_kg, "kg",
        "Conversion budget and selected loading"
    ),
    (
        "Equivalent acceleration mass",
        derived["equivalent_mass_kg"], "kg",
        "Three assumed solid-cylinder wheels"
    ),
    (
        "Front static load",
        current_front_load_N, "N",
        "Depends on assumed CG position"
    ),
    (
        "Combined rear static load",
        current_rear_load_N, "N",
        "Depends on assumed CG position"
    ),
    (
        "Nominal battery energy",
        assumed["voltage_V"] * assumed["capacity_Ah"] / 1000,
        "kWh",
        "Capacity is provisional; not usable energy"
    ),
    (
        "Wheel power at 30 km/h",
        baseline_30kmh["wheel_power_W"] / 1000,
        "kW",
        "Steady level-road requirement"
    ),
    (
        "Speed corresponding to maximum motor RPM",
        derived["speed_at_max_motor_rpm_kmh"],
        "km/h",
        "Kinematic reference; not achievable-speed prediction"
    ),
], columns=["Quantity", "Value", "Unit", "Interpretation"])

display(baseline_summary.round({"Value": 3}))

# ------------------------------------------------------------
# 3. RECORD MODEL READINESS
# ------------------------------------------------------------

subsystem_status = pd.DataFrame([
    ("Vehicle configuration", "Established provisionally"),
    ("3D packaging layout", "Implemented; assumed component geometry"),
    ("Static axle loads", "Implemented; assumed CG"),
    ("Steady road resistance", "Implemented"),
    ("Battery dynamics / BMS", "Not yet modelled"),
    ("Inverter dynamics / losses", "Not yet modelled"),
    ("Motor dynamics / losses", "Not yet modelled"),
    ("Transient vehicle dynamics", "Not yet modelled"),
    ("Regenerative braking", "Not yet modelled"),
    ("Thermal behaviour", "Not yet modelled"),
    ("Physical validation", "Not performed"),
    ("PINN estimation", "Deferred"),
], columns=["Subsystem / activity", "Status"])

display(subsystem_status)

# ------------------------------------------------------------
# 4. CREATE AN INDEPENDENT CONFIGURATION SNAPSHOT
# ------------------------------------------------------------

baseline_record = deepcopy({
    "baseline_version": "0.1",
    "status": "provisional_reference_configuration",
    "physical_validation": "not_performed",
    "vehicle_geometry": vehicle_geometry,
    "vehicle_reference": vehicle_reference,
    "motor_specs": motor_specs,
    "thesis_model": thesis_model,
    "mass_budget": mass_budget,
    "assumed": assumed,
    "derived": derived,
    "environment": environment,
    "architecture": architecture,
    "model_settings": model_settings,
    "layout_assumptions": layout_assumptions,
    "packaging": packaging,
    "parameter_register": parameter_register.to_dict(
        orient="records"
    ),
    "baseline_summary": baseline_summary.to_dict(
        orient="records"
    ),
    "limitations": [
        "Converted vehicle mass is based on an assumed budget.",
        "Battery selection and capacity remain provisional.",
        "CG position and component placement are assumed.",
        "3D layout is not exact exterior CAD.",
        "No full collision or clearance assessment performed.",
        "Motor overload duration and rating conventions unverified.",
        "Steady road-load calculations do not prove feasibility.",
        "No physical vehicle validation performed.",
    ],
})

# Validate serialization before writing the checkpoint.
baseline_json = json.dumps(
    baseline_record,
    indent=2,
    allow_nan=False
)

checkpoint_filename = "electric_tricycle_baseline_v0_1.json"

with open(
    checkpoint_filename, "w", encoding="utf-8"
) as file:
    file.write(baseline_json)

print("PASS: Stage 1 configuration review completed.")
print("Checkpoint created:", checkpoint_filename)
print("Location: the Colab runtime Files panel.")
print("Download the checkpoint if you want to retain it.")
print("No network upload or public sharing is performed by this cell.")

,Check,Result
0,Loading within gross-mass limit,PASS
1,CG between front and rear axles,PASS
2,Positive static axle loads,PASS
3,Static axle-load balance,PASS
4,Valid SOC operating window,PASS
5,Aerodynamic inputs consistent,PASS
6,Thesis lumped-loss mode selected,PASS
7,PINN work deferred,PASS
8,Previous calculations match current mass,PASS
9,Previous wheel-load calculation is current,PASS


,Quantity,Value,Unit,Interpretation
0,Total physical mass,645.000,kg,Conversion budget and selected loading
1,Equivalent acceleration mass,663.000,kg,Three assumed solid-cylinder wheels
2,Front static load,2306.883,N,Depends on assumed CG position
3,Combined rear static load,4020.567,N,Depends on assumed CG position
4,Nominal battery energy,7.200,kWh,Capacity is provisional; not usable energy
5,Wheel power at 30 km/h,1.120,kW,Steady level-road requirement
6,Speed corresponding to maximum motor RPM,57.012,km/h,Kinematic reference; not achievable-speed pred...


,Subsystem / activity,Status
0,Vehicle configuration,Established provisionally
1,3D packaging layout,Implemented; assumed component geometry
2,Static axle loads,Implemented; assumed CG
3,Steady road resistance,Implemented
4,Battery dynamics / BMS,Not yet modelled
5,Inverter dynamics / losses,Not yet modelled
6,Motor dynamics / losses,Not yet modelled
7,Transient vehicle dynamics,Not yet modelled
8,Regenerative braking,Not yet modelled
9,Thermal behaviour,Not yet modelled


PASS: Stage 1 configuration review completed.
Checkpoint created: electric_tricycle_baseline_v0_1.json
Location: the Colab runtime Files panel.
Download the checkpoint if you want to retain it.
No network upload or public sharing is performed by this cell.


In [8]:
# STAGE 1 — CELL 4
# Final baseline checks, summary and configuration checkpoint.
# No dynamic simulation or PINN training.

import json
from copy import deepcopy

required_objects = [
    "vehicle_geometry", "vehicle_reference", "motor_specs",
    "thesis_model", "mass_budget", "assumed", "derived",
    "environment", "architecture", "model_settings",
    "packaging", "layout_assumptions", "parameter_register",
    "calculate_steady_road_load", "np", "pd", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted Stage 1 Cells 1–3 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------…
# STAGE 2 — CELL 5
# Battery arrangement and ideal constant-current SOC model.
# Positive current = discharge; negative current = charge.
# No terminal-voltage, thermal or BMS dynamics yet.

required_objects = [
    "assumed", "model_settings", "baseline_checks",
    "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted Stage 1 Cells 1–4 first. Missing: "
        + ", ".join(missing_objects)
    )

if not all(bool(passed) for _, passed in baseline_checks):
    raise RuntimeError(
        "Resolve the Stage 1 baseline failures before continuing."
    )

# ------------------------------------------------------------
# 1. BATTERY PARAMETERS
# New dictionary: battery_params holds battery-specific inputs.
# Existing nominal pack voltage/capacity remain in assumed.
# ------------------------------------------------------------

battery_params = {
    "series_cells": 15,
    "parallel_strings": 1,
    "cell_nominal_voltage_V": 3.2,
    "cell_capacity_Ah": 150.0,
    "capacity_soh": 1.0,
    "parameter_evidence": "provisional simulation assumptions",
}

battery_derived = {
    "total_cells": (
        battery_params["series_cells"]
        * battery_params["parallel_strings"]
    ),
    "nominal_voltage_V": (
        battery_params["series_cells"]
        * battery_params["cell_nominal_voltage_V"]
    ),
    "nominal_capacity_Ah": (
        battery_params["parallel_strings"]
        * battery_params["cell_capacity_Ah"]
    ),
}

battery_derived["available_capacity_Ah"] = (
    battery_derived["nominal_capacity_Ah"]
    * battery_params["capacity_soh"]
)

assert isinstance(battery_params["series_cells"], int)
assert isinstance(battery_params["parallel_strings"], int)
assert battery_params["series_cells"] > 0
assert battery_params["parallel_strings"] > 0
assert battery_params["cell_nominal_voltage_V"] > 0
assert battery_params["cell_capacity_Ah"] > 0
assert 0 < battery_params["capacity_soh"] <= 1

assert np.isclose(
    battery_derived["nominal_voltage_V"],
    assumed["voltage_V"]
), "Cell arrangement conflicts with baseline nominal voltage."

assert np.isclose(
    battery_derived["nominal_capacity_Ah"],
    assumed["capacity_Ah"]
), "Cell arrangement conflicts with baseline nominal capacity."

# ------------------------------------------------------------
# 2. IDEAL CONSTANT-CURRENT SIMULATION
# ------------------------------------------------------------

def simulate_battery_soc_constant_current(
    current_A,
    duration_s,
    dt_s=1.0,
):
    """
    Calculate SOC under imposed constant current.

    Assumptions:
    - Fixed available capacity and SOH.
    - Ideal charge accounting.
    - No self-discharge, voltage or temperature model.
    - No BMS action.

    Tests crossing the selected SOC window are rejected.
    SOC is not silently clipped.
    """
    inputs = np.array(
        [current_A, duration_s, dt_s], dtype=float
    )

    if not np.all(np.isfinite(inputs)):
        raise ValueError("Inputs must be finite.")

    if duration_s <= 0 or dt_s <= 0:
        raise ValueError("Duration and timestep must be positive.")

    if not 0 < battery_params["capacity_soh"] <= 1:
        raise ValueError("Capacity SOH must be in (0, 1].")

    if not (
        0 <= assumed["minimum_soc"]
        <= assumed["initial_soc"]
        <= assumed["maximum_soc"] <= 1
    ):
        raise ValueError("Invalid initial SOC or operating window.")

    # Verify cell-level capacity still matches the active baseline.
    nominal_capacity_Ah = (
        battery_params["parallel_strings"]
        * battery_params["cell_capacity_Ah"]
    )

    if not np.isclose(
        nominal_capacity_Ah, assumed["capacity_Ah"]
    ):
        raise ValueError(
            "Battery cell capacity and baseline capacity disagree."
        )

    available_capacity_Ah = (
        nominal_capacity_Ah * battery_params["capacity_soh"]
    )

    if available_capacity_Ah <= 0:
        raise ValueError("Available capacity must be positive.")

    # Include the exact endpoint, including a partial last interval.
    time_s = np.arange(0.0, duration_s, dt_s)
    time_s = np.append(time_s, float(duration_s))

    # Positive = net charge removed; negative = net charge added.
    net_charge_removed_Ah = current_A * time_s / 3600.0

    soc = (
        assumed["initial_soc"]
        - net_charge_removed_Ah / available_capacity_Ah
    )

    tolerance = 1e-12
    if (
        soc.min() < assumed["minimum_soc"] - tolerance
        or soc.max() > assumed["maximum_soc"] + tolerance
    ):
        raise ValueError(
            "The requested test crosses the selected SOC window. "
            "Reduce its current or duration. "
            "BMS cutoff is not implemented yet."
        )

    return pd.DataFrame({
        "time_s": time_s,
        "pack_current_A": np.full(time_s.shape, current_A),
        "SOC": soc,
        "net_charge_removed_Ah": net_charge_removed_Ah,
    })

# ------------------------------------------------------------
# 3. TEST CASES
# ------------------------------------------------------------

battery_soc_result = simulate_battery_soc_constant_current(
    current_A=30.0,
    duration_s=600.0,
    dt_s=1.0,
)

battery_rest_check = simulate_battery_soc_constant_current(
    current_A=0.0,
    duration_s=600.0,
)

# Charge-accounting test only; charging safety is not established.
battery_charge_check = simulate_battery_soc_constant_current(
    current_A=-30.0,
    duration_s=60.0,
)

expected_charge_removed_Ah = 30.0 * 600.0 / 3600.0

expected_final_soc = (
    assumed["initial_soc"]
    - expected_charge_removed_Ah
    / battery_derived["available_capacity_Ah"]
)

assert np.isclose(
    battery_soc_result["SOC"].iloc[-1],
    expected_final_soc
)

assert np.allclose(
    battery_rest_check["SOC"],
    assumed["initial_soc"]
)

assert (
    battery_charge_check["SOC"].iloc[-1]
    > assumed["initial_soc"]
)

assert np.all(np.diff(battery_soc_result["SOC"]) <= 0)

# ------------------------------------------------------------
# 4. RESULTS
# ------------------------------------------------------------

battery_summary = pd.DataFrame([
    ("Series cells", battery_params["series_cells"], "cells"),
    ("Parallel strings",
     battery_params["parallel_strings"], "strings"),
    ("Nominal pack voltage",
     battery_derived["nominal_voltage_V"], "V"),
    ("Available capacity",
     battery_derived["available_capacity_Ah"], "Ah"),
    ("Charge removed",
     battery_soc_result["net_charge_removed_Ah"].iloc[-1], "Ah"),
    ("Initial SOC", assumed["initial_soc"] * 100.0, "%"),
    ("Final SOC",
     battery_soc_result["SOC"].iloc[-1] * 100.0, "%"),
], columns=["Quantity", "Value", "Unit"])

display(battery_summary.round(3))

battery_soc_figure = go.Figure()

battery_soc_figure.add_trace(go.Scatter(
    x=battery_soc_result["time_s"] / 60.0,
    y=battery_soc_result["SOC"] * 100.0,
    mode="lines",
    name="Simulated SOC",
    line=dict(color="#27ae60", width=3),
))

battery_soc_figure.update_layout(
    title="Ideal battery charge accounting | 30 A discharge",
    xaxis_title="Time [min]",
    yaxis_title="SOC [%]",
    template="plotly_white",
    height=420,
)

battery_soc_figure.show()

print("PASS: battery arrangement matches the baseline.")
print("PASS: constant-current charge accounting.")
print("PASS: zero-current SOC remains constant.")
print("PASS: charging current increases SOC.")
print("PASS: discharge SOC decreases monotonically.")
print("These are mathematical checks, not physical validation.")

,Quantity,Value,Unit
0,Series cells,15.000,cells
1,Parallel strings,1.000,strings
2,Nominal pack voltage,48.000,V
3,Available capacity,150.000,Ah
4,Charge removed,5.000,Ah
5,Initial SOC,80.000,%
6,Final SOC,76.667,%


PASS: battery arrangement matches the baseline.
PASS: constant-current charge accounting.
PASS: zero-current SOC remains constant.
PASS: charging current increases SOC.
PASS: discharge SOC decreases monotonically.
These are mathematical checks, not physical validation.


In [9]:
# STAGE 2 — CELL 6
# Resistance-only battery voltage, power and heat-generation model.
# OCV curve and resistance are synthetic assumptions.
# No polarization, temperature or BMS dynamics yet.

required_objects = [
    "battery_params", "battery_soc_result",
    "assumed", "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted notebook through Cell 5 first. Missing: "
        + ", ".join(missing_objects)
    )

# ------------------------------------------------------------
# 1. ADD ELECTRICAL PARAMETERS
# Existing values are preserved if this cell is rerun.
# ------------------------------------------------------------

battery_params.setdefault("R0_pack_ohm", 0.012)

battery_params.setdefault(
    "ocv_soc_points",
    [0.0, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0]
)

battery_params.setdefault(
    "ocv_cell_points_V",
    [2.90, 3.15, 3.22, 3.27, 3.30, 3.33, 3.36, 3.40]
)

battery_params["ocv_parameter_evidence"] = (
    "Synthetic illustrative curve; not measured cell data"
)

battery_params["R0_parameter_evidence"] = (
    "Assumed constant pack-level resistance"
)

# ------------------------------------------------------------
# 2. OCV FUNCTION
# ------------------------------------------------------------

def battery_ocv_pack(soc):
    """
    Interpolate the synthetic cell OCV curve and scale to pack level.
    Values outside the curve domain are rejected, not extrapolated.
    """
    soc = np.asarray(soc, dtype=float)

    soc_points = np.asarray(
        battery_params["ocv_soc_points"], dtype=float
    )
    cell_voltage_points = np.asarray(
        battery_params["ocv_cell_points_V"], dtype=float
    )

    if (
        soc_points.ndim != 1
        or cell_voltage_points.ndim != 1
        or len(soc_points) < 2
        or len(soc_points) != len(cell_voltage_points)
    ):
        raise ValueError("Invalid OCV table dimensions.")

    if not (
        np.all(np.isfinite(soc_points))
        and np.all(np.isfinite(cell_voltage_points))
        and np.all(np.diff(soc_points) > 0)
        and np.all(cell_voltage_points > 0)
    ):
        raise ValueError("OCV table contains invalid values.")

    if not np.all(np.isfinite(soc)):
        raise ValueError("SOC must contain finite values.")

    if (
        np.any(soc < soc_points[0])
        or np.any(soc > soc_points[-1])
    ):
        raise ValueError("SOC is outside the OCV table domain.")

    cell_ocv_V = np.interp(
        soc, soc_points, cell_voltage_points
    )

    return battery_params["series_cells"] * cell_ocv_V

# ------------------------------------------------------------
# 3. RESISTANCE-ONLY ELECTRICAL FUNCTION
# ------------------------------------------------------------

def calculate_battery_electrical(soc, current_A):
    """
    Calculate instantaneous electrical quantities.

    Positive current: battery supplies terminal power.
    Negative current: battery receives terminal power.

    This function does not enforce BMS or inverter limits.
    """
    soc, current_A = np.broadcast_arrays(
        np.asarray(soc, dtype=float),
        np.asarray(current_A, dtype=float),
    )

    if not np.all(np.isfinite(current_A)):
        raise ValueError("Current must contain finite values.")

    R0 = float(battery_params["R0_pack_ohm"])

    if not np.isfinite(R0) or R0 < 0:
        raise ValueError("Pack resistance must be finite and nonnegative.")

    ocv_V = battery_ocv_pack(soc)
    terminal_voltage_V = ocv_V - current_A * R0

    if np.any(terminal_voltage_V <= 0):
        raise ValueError(
            "Nonpositive terminal voltage: imposed current is "
            "outside this simplified model's useful operating range."
        )

    return {
        "ocv_V": ocv_V,
        "terminal_voltage_V": terminal_voltage_V,
        "terminal_power_W": terminal_voltage_V * current_A,
        "ohmic_heat_W": current_A**2 * R0,
        "ocv_power_W": ocv_V * current_A,
    }

# ------------------------------------------------------------
# 4. APPLY TO THE EXISTING SOC TEST
# Preserve battery_soc_result as the original charge-accounting result.
# ------------------------------------------------------------

battery_electrical_result = battery_soc_result.copy()

electrical_values = calculate_battery_electrical(
    battery_electrical_result["SOC"].to_numpy(),
    battery_electrical_result["pack_current_A"].to_numpy(),
)

for name, values in electrical_values.items():
    battery_electrical_result[name] = values

# ------------------------------------------------------------
# 5. ENGINEERING CHECKS
# ------------------------------------------------------------

assert np.allclose(
    battery_electrical_result["ocv_power_W"],
    battery_electrical_result["terminal_power_W"]
    + battery_electrical_result["ohmic_heat_W"],
)

rest_electrical_check = calculate_battery_electrical(
    soc=assumed["initial_soc"],
    current_A=0.0,
)

assert np.isclose(
    rest_electrical_check["terminal_voltage_V"],
    rest_electrical_check["ocv_V"],
)

assert np.isclose(
    rest_electrical_check["ohmic_heat_W"], 0.0
)

discharge_electrical_check = calculate_battery_electrical(
    soc=assumed["initial_soc"],
    current_A=30.0,
)

charge_electrical_check = calculate_battery_electrical(
    soc=assumed["initial_soc"],
    current_A=-30.0,
)

assert (
    discharge_electrical_check["terminal_voltage_V"]
    < discharge_electrical_check["ocv_V"]
)

assert (
    charge_electrical_check["terminal_voltage_V"]
    > charge_electrical_check["ocv_V"]
)

assert np.isclose(
    discharge_electrical_check["ohmic_heat_W"],
    charge_electrical_check["ohmic_heat_W"],
)

# ------------------------------------------------------------
# 6. TABLE AND GRAPHS
# ------------------------------------------------------------

display(
    battery_electrical_result.iloc[[0, -1]][[
        "time_s", "SOC", "pack_current_A",
        "ocv_V", "terminal_voltage_V",
        "terminal_power_W", "ohmic_heat_W"
    ]].round(4)
)

ocv_figure = go.Figure()

ocv_figure.add_trace(go.Scatter(
    x=np.asarray(battery_params["ocv_soc_points"]) * 100.0,
    y=(
        np.asarray(battery_params["ocv_cell_points_V"])
        * battery_params["series_cells"]
    ),
    mode="lines+markers",
    name="Assumed pack OCV",
))

ocv_figure.update_layout(
    title="Synthetic OCV–SOC curve | not measured cell data",
    xaxis_title="SOC [%]",
    yaxis_title="Pack open-circuit voltage [V]",
    template="plotly_white",
    height=380,
)

ocv_figure.show()

battery_voltage_figure = go.Figure()

for column, label in [
    ("ocv_V", "Open-circuit voltage"),
    ("terminal_voltage_V", "Terminal voltage under load"),
]:
    battery_voltage_figure.add_trace(go.Scatter(
        x=battery_electrical_result["time_s"] / 60.0,
        y=battery_electrical_result[column],
        name=label,
        mode="lines",
    ))

battery_voltage_figure.update_layout(
    title="Battery voltage | existing constant-current discharge test",
    xaxis_title="Time [min]",
    yaxis_title="Voltage [V]",
    template="plotly_white",
    height=400,
)

battery_voltage_figure.show()

print("PASS: instantaneous circuit power balance.")
print("PASS: zero-current terminal voltage equals OCV.")
print("PASS: zero-current ohmic heating is zero.")
print("PASS: discharge/charge voltage directions are correct.")
print("PASS: equal-magnitude currents give equal ohmic heating.")
print("No temperature, BMS or hardware compatibility validation yet.")

,time_s,SOC,pack_current_A,ocv_V,terminal_voltage_V,terminal_power_W,ohmic_heat_W
0,0.0,0.8000,30.0,49.950,49.590,1487.70,10.8
600,600.0,0.7667,30.0,49.875,49.515,1485.45,10.8


PASS: instantaneous circuit power balance.
PASS: zero-current terminal voltage equals OCV.
PASS: zero-current ohmic heating is zero.
PASS: discharge/charge voltage directions are correct.
PASS: equal-magnitude currents give equal ohmic heating.
No temperature, BMS or hardware compatibility validation yet.


In [12]:
# DIAGNOSTIC FOR CELL 7 — does not change the model.

required_columns = [
    "time_s",
    "pack_current_A",
    "ocv_V",
    "terminal_voltage_V",
    "terminal_power_W",
    "resistive_heat_W",
    "rc_energy_rate_W",
    "polarization_V",
]

if "battery_rc_result" not in globals():
    raise RuntimeError(
        "battery_rc_result is missing. Run Cell 7 first."
    )

missing_columns = [
    name for name in required_columns
    if name not in battery_rc_result.columns
]

if missing_columns:
    raise RuntimeError(
        "Missing result columns: " + ", ".join(missing_columns)
    )

diagnostic = battery_rc_result[required_columns].copy()

diagnostic["source_power_W"] = (
    diagnostic["ocv_V"] * diagnostic["pack_current_A"]
)

diagnostic["accounted_power_W"] = (
    diagnostic["terminal_power_W"]
    + diagnostic["resistive_heat_W"]
    + diagnostic["rc_energy_rate_W"]
)

diagnostic["balance_error_W"] = (
    diagnostic["source_power_W"]
    - diagnostic["accounted_power_W"]
)

numeric_values = diagnostic.to_numpy(dtype=float)
finite_rows = np.isfinite(numeric_values).all(axis=1)

print("Rows containing NaN or infinity:", int((~finite_rows).sum()))

if finite_rows.any():
    finite_errors = diagnostic.loc[
        finite_rows, "balance_error_W"
    ].abs()

    print(
        "Maximum absolute power-balance error [W]:",
        finite_errors.max()
    )

    worst_indices = finite_errors.nlargest(5).index

    print("\nRows with the largest finite errors:")
    display(diagnostic.loc[worst_indices])

if not finite_rows.all():
    print("\nFirst rows containing invalid numbers:")
    display(diagnostic.loc[~finite_rows].head())

print("\nCurrent electrical parameters:")
for name in ["R0_pack_ohm", "R1_pack_ohm", "C1_pack_F"]:
    print(name, "=", battery_params.get(name, "MISSING"))

Rows containing NaN or infinity: 0
Maximum absolute power-balance error [W]: 47.84259961475954

Rows with the largest finite errors:


,time_s,pack_current_A,ocv_V,terminal_voltage_V,terminal_power_W,resistive_heat_W,rc_energy_rate_W,polarization_V,source_power_W,accounted_power_W,balance_error_W
180,180.0,0.0,49.935,49.779360,0.0,51.879883,-4.037283,0.155640,0.0,47.842600,-47.842600
181,181.0,0.0,49.935,49.781933,0.0,51.022384,-3.904925,0.153067,0.0,47.117458,-47.117458
182,182.0,0.0,49.935,49.784463,0.0,50.179058,-3.776907,0.150537,0.0,46.402151,-46.402151
183,183.0,0.0,49.935,49.786951,0.0,49.349671,-3.653085,0.148049,0.0,45.696586,-45.696586
184,184.0,0.0,49.935,49.789398,0.0,48.533993,-3.533323,0.145602,0.0,45.000670,-45.000670



Current electrical parameters:
R0_pack_ohm = 0.012
R1_pack_ohm = 0.006
C1_pack_F = 10000.0


In [13]:
# STAGE 2 — CELL 7
# Complete one-RC battery model with corrected power balance.
# Positive current = discharge.
# Parameters remain provisional assumptions.

required_objects = [
    "battery_params", "battery_ocv_pack",
    "assumed", "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted notebook through Cell 6 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

# Preserve existing values when this cell is rerun.
battery_params.setdefault("R1_pack_ohm", 0.006)
battery_params.setdefault("C1_pack_F", 10000.0)
battery_params.setdefault("initial_polarization_V", 0.0)


def simulate_battery_one_rc(time_s, current_A):
    """
    One-RC battery equivalent-circuit simulation.

    Current convention:
        Positive = discharge.
        Negative = charge.

    Sampling convention:
        current_A[k] applies over [time_s[k], time_s[k+1]).
        The final current sample determines endpoint voltage only.

    Includes:
        SOC, OCV, terminal voltage, polarization,
        terminal power, resistor heating and RC energy storage.

    Excludes:
        Temperature feedback, BMS actions and ageing.
    """
    time_s = np.asarray(time_s, dtype=float)
    current_A = np.asarray(current_A, dtype=float)

    # --------------------------------------------------------
    # 1. VALIDATE INPUTS
    # --------------------------------------------------------

    if (
        time_s.ndim != 1
        or current_A.ndim != 1
        or time_s.shape != current_A.shape
        or len(time_s) < 2
    ):
        raise ValueError(
            "Time and current must be matching 1D arrays "
            "with at least two samples."
        )

    if not (
        np.all(np.isfinite(time_s))
        and np.all(np.isfinite(current_A))
        and np.all(np.diff(time_s) > 0)
    ):
        raise ValueError(
            "Inputs must be finite and time must increase."
        )

    R0 = float(battery_params["R0_pack_ohm"])
    R1 = float(battery_params["R1_pack_ohm"])
    C1 = float(battery_params["C1_pack_F"])

    if not (
        np.all(np.isfinite([R0, R1, C1]))
        and R0 >= 0
        and R1 > 0
        and C1 > 0
    ):
        raise ValueError("Invalid resistance or capacitance.")

    nominal_capacity_Ah = (
        battery_params["parallel_strings"]
        * battery_params["cell_capacity_Ah"]
    )

    if not np.isclose(
        nominal_capacity_Ah,
        assumed["capacity_Ah"]
    ):
        raise ValueError(
            "Battery cell capacity and baseline capacity disagree."
        )

    capacity_soh = float(battery_params["capacity_soh"])

    if not np.isfinite(capacity_soh) or not 0 < capacity_soh <= 1:
        raise ValueError("Capacity SOH must be in (0, 1].")

    available_capacity_Ah = nominal_capacity_Ah * capacity_soh

    if (
        not np.isfinite(available_capacity_Ah)
        or available_capacity_Ah <= 0
    ):
        raise ValueError("Available capacity must be positive.")

    if not (
        0 <= assumed["minimum_soc"]
        <= assumed["initial_soc"]
        <= assumed["maximum_soc"] <= 1
    ):
        raise ValueError("Invalid SOC operating window.")

    initial_polarization_V = float(
        battery_params["initial_polarization_V"]
    )

    if not np.isfinite(initial_polarization_V):
        raise ValueError("Initial polarization must be finite.")

    # --------------------------------------------------------
    # 2. INITIAL CONDITIONS
    # --------------------------------------------------------

    tau_s = R1 * C1

    soc = np.empty(len(time_s), dtype=float)
    polarization_V = np.empty(len(time_s), dtype=float)

    soc[0] = assumed["initial_soc"]
    polarization_V[0] = initial_polarization_V

    # --------------------------------------------------------
    # 3. EXACT INTERVAL UPDATES
    # Current is constant within each interval.
    # --------------------------------------------------------

    for k, dt_s in enumerate(np.diff(time_s)):
        decay = np.exp(-dt_s / tau_s)
        one_minus_decay = -np.expm1(-dt_s / tau_s)

        soc[k + 1] = (
            soc[k]
            - current_A[k] * dt_s
            / (3600.0 * available_capacity_Ah)
        )

        polarization_V[k + 1] = (
            polarization_V[k] * decay
            + current_A[k] * R1 * one_minus_decay
        )

        if not (
            assumed["minimum_soc"] - 1e-12
            <= soc[k + 1]
            <= assumed["maximum_soc"] + 1e-12
        ):
            raise ValueError(
                "Test crosses the selected SOC window. "
                "Reduce current or duration; "
                "BMS cutoff is not implemented yet."
            )

    # --------------------------------------------------------
    # 4. VOLTAGE AND POWER
    # --------------------------------------------------------

    ocv_V = battery_ocv_pack(soc)

    terminal_voltage_V = (
        ocv_V - current_A * R0 - polarization_V
    )

    if not np.all(np.isfinite(terminal_voltage_V)):
        raise ValueError("Terminal voltage contains invalid values.")

    if np.any(terminal_voltage_V <= 0):
        raise ValueError("Nonpositive terminal voltage.")

    terminal_power_W = terminal_voltage_V * current_A

    # Correct resistor dissipation:
    # R0 heating = I_pack² * R0
    # R1 heating = V1² / R1
    ohmic_heat_W = np.square(current_A) * R0
    polarization_heat_W = np.square(polarization_V) / R1

    resistive_heat_W = (
        ohmic_heat_W + polarization_heat_W
    )

    # Equivalent capacitor energy and its instantaneous derivative:
    # E_RC = 0.5 * C1 * V1²
    # dE_RC/dt = V1 * I_pack - V1² / R1
    rc_stored_energy_J = (
        0.5 * C1 * np.square(polarization_V)
    )

    rc_energy_rate_W = (
        polarization_V * current_A
        - polarization_heat_W
    )

    ocv_power_W = ocv_V * current_A

    power_balance_error_W = (
        ocv_power_W
        - terminal_power_W
        - resistive_heat_W
        - rc_energy_rate_W
    )

    return pd.DataFrame({
        "time_s": time_s,
        "pack_current_A": current_A,
        "SOC": soc,
        "polarization_V": polarization_V,
        "ocv_V": ocv_V,
        "terminal_voltage_V": terminal_voltage_V,
        "terminal_power_W": terminal_power_W,
        "ohmic_heat_W": ohmic_heat_W,
        "polarization_heat_W": polarization_heat_W,
        "resistive_heat_W": resistive_heat_W,
        "rc_stored_energy_J": rc_stored_energy_J,
        "rc_energy_rate_W": rc_energy_rate_W,
        "ocv_power_W": ocv_power_W,
        "power_balance_error_W": power_balance_error_W,
    })


# ------------------------------------------------------------
# 5. PULSE-AND-REST TEST
# ------------------------------------------------------------

pulse_time_s = np.arange(0.0, 361.0, 1.0)

pulse_current_A = np.where(
    (pulse_time_s >= 60.0) & (pulse_time_s < 180.0),
    30.0,
    0.0,
)

battery_rc_result = simulate_battery_one_rc(
    pulse_time_s,
    pulse_current_A,
)

# ------------------------------------------------------------
# 6. ANALYTICAL CHECKS
# ------------------------------------------------------------

R1 = battery_params["R1_pack_ohm"]
C1 = battery_params["C1_pack_F"]
tau_s = R1 * C1

# Includes any nonzero initial polarization already configured.
expected_polarization_180_V = (
    battery_params["initial_polarization_V"]
    * np.exp(-180.0 / tau_s)
    + 30.0 * R1 * (1.0 - np.exp(-120.0 / tau_s))
)

assert np.isclose(
    battery_rc_result.loc[180, "polarization_V"],
    expected_polarization_180_V,
), "Analytical pulse-response check failed."

assert np.isclose(
    battery_rc_result.loc[360, "polarization_V"],
    expected_polarization_180_V * np.exp(-180.0 / tau_s),
), "Analytical relaxation check failed."

available_capacity_Ah = (
    assumed["capacity_Ah"]
    * battery_params["capacity_soh"]
)

expected_final_soc = (
    assumed["initial_soc"]
    - (30.0 * 120.0 / 3600.0) / available_capacity_Ah
)

assert np.isclose(
    battery_rc_result["SOC"].iloc[-1],
    expected_final_soc,
), "Charge-accounting check failed."

assert np.allclose(
    battery_rc_result.loc[180:, "SOC"],
    battery_rc_result.loc[180, "SOC"],
), "SOC changed during zero-current rest."

assert np.all(
    battery_rc_result["resistive_heat_W"] >= 0
), "Negative resistor heating detected."

maximum_balance_error_W = float(
    battery_rc_result["power_balance_error_W"].abs().max()
)

assert np.isfinite(maximum_balance_error_W)
assert maximum_balance_error_W <= 1e-9, (
    "Circuit power-balance check failed. "
    f"Maximum error: {maximum_balance_error_W:.6e} W"
)

# At rest, RC energy release must equal resistor heating.
rest_rows = battery_rc_result["pack_current_A"] == 0.0

assert np.allclose(
    battery_rc_result.loc[rest_rows, "resistive_heat_W"],
    -battery_rc_result.loc[rest_rows, "rc_energy_rate_W"],
    atol=1e-9,
), "Rest-period energy balance failed."

# ------------------------------------------------------------
# 7. TIMESTEP REFINEMENT
# Pulse transitions remain aligned in both simulations.
# ------------------------------------------------------------

fine_time_s = np.arange(0.0, 360.5, 0.5)

fine_current_A = np.where(
    (fine_time_s >= 60.0) & (fine_time_s < 180.0),
    30.0,
    0.0,
)

battery_rc_refined = simulate_battery_one_rc(
    fine_time_s,
    fine_current_A,
)

assert np.allclose(
    battery_rc_result[
        ["SOC", "polarization_V"]
    ].to_numpy(),
    battery_rc_refined.iloc[::2][
        ["SOC", "polarization_V"]
    ].to_numpy(),
    atol=1e-10,
), "Aligned timestep-refinement check failed."

# ------------------------------------------------------------
# 8. RESULTS TABLE
# ------------------------------------------------------------

display(
    battery_rc_result.loc[
        [0, 60, 120, 180, 240, 360],
        [
            "time_s",
            "pack_current_A",
            "SOC",
            "polarization_V",
            "terminal_voltage_V",
            "resistive_heat_W",
            "rc_energy_rate_W",
        ],
    ].round(6)
)

# ------------------------------------------------------------
# 9. GRAPHS
# ------------------------------------------------------------

battery_rc_figure = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    subplot_titles=[
        "Imposed battery current",
        "Terminal voltage and open-circuit voltage",
        "Polarization voltage",
    ],
    vertical_spacing=0.09,
)

battery_rc_figure.add_trace(
    go.Scatter(
        x=pulse_time_s,
        y=pulse_current_A,
        name="Current",
        line_shape="hv",
    ),
    row=1, col=1,
)

for column, label in [
    ("terminal_voltage_V", "Terminal voltage"),
    ("ocv_V", "OCV"),
]:
    battery_rc_figure.add_trace(
        go.Scatter(
            x=pulse_time_s,
            y=battery_rc_result[column],
            name=label,
        ),
        row=2, col=1,
    )

battery_rc_figure.add_trace(
    go.Scatter(
        x=pulse_time_s,
        y=battery_rc_result["polarization_V"],
        name="Polarization voltage",
    ),
    row=3, col=1,
)

battery_rc_figure.update_yaxes(
    title_text="Current [A]", row=1, col=1
)
battery_rc_figure.update_yaxes(
    title_text="Voltage [V]", row=2, col=1
)
battery_rc_figure.update_yaxes(
    title_text="Voltage [V]", row=3, col=1
)
battery_rc_figure.update_xaxes(
    title_text="Time [s]", row=3, col=1
)

battery_rc_figure.update_layout(
    title="Assumed one-RC battery | pulse discharge and recovery",
    template="plotly_white",
    height=800,
)

battery_rc_figure.show()

print(f"Polarization time constant: {tau_s:.3f} s")
print(
    "Maximum circuit power-balance error: "
    f"{maximum_balance_error_W:.3e} W"
)
print("PASS: analytical pulse and relaxation responses.")
print("PASS: charge accounting and constant rest SOC.")
print("PASS: nonnegative resistor heating.")
print("PASS: circuit power balance including RC storage.")
print("PASS: rest-period energy balance.")
print("PASS: aligned timestep-refinement check.")
print("Parameters remain assumed; no physical validation claimed.")

,time_s,pack_current_A,SOC,polarization_V,terminal_voltage_V,resistive_heat_W,rc_energy_rate_W
0,0.0,0.0,0.800000,0.000000,49.950000,0.000000,0.000000
60,60.0,30.0,0.800000,0.000000,49.590000,10.800000,0.000000
120,120.0,30.0,0.796667,0.113782,49.468718,12.957713,1.255738
180,180.0,0.0,0.793333,0.155640,49.779360,4.037283,-4.037283
240,240.0,0.0,0.793333,0.057257,49.877743,0.546387,-0.546387
360,360.0,0.0,0.793333,0.007749,49.927251,0.010007,-0.010007


Polarization time constant: 60.000 s
Maximum circuit power-balance error: 2.558e-13 W
PASS: analytical pulse and relaxation responses.
PASS: charge accounting and constant rest SOC.
PASS: nonnegative resistor heating.
PASS: circuit power balance including RC storage.
PASS: rest-period energy balance.
PASS: aligned timestep-refinement check.
Parameters remain assumed; no physical validation claimed.


In [14]:
# STAGE 2 — CELL 8
# Lumped battery thermal model driven by Cell 7 electrical losses.
# One-way electrical-to-thermal coupling.
# No BMS temperature protection yet.

required_objects = [
    "battery_params", "battery_rc_result",
    "environment", "np", "pd", "go", "display"
]

missing_objects = [
    name for name in required_objects if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted notebook through Cell 7 first. Missing: "
        + ", ".join(missing_objects)
    )

# New battery-specific thermal parameters.
# These values are assumptions, not identified measurements.
battery_params.setdefault("thermal_capacity_J_per_K", 60000.0)
battery_params.setdefault("thermal_conductance_W_per_K", 15.0)
battery_params.setdefault(
    "initial_temperature_C",
    environment["ambient_temperature_C"]
)

def simulate_battery_temperature(
    time_s,
    heat_W,
    ambient_temperature_C,
    initial_temperature_C,
):
    """
    Uniform battery-temperature model.

    heat_W[k] applies over [time_s[k], time_s[k+1]).
    Ambient temperature is constant for this test.
    The final heat sample does not affect the final temperature.
    """
    time_s = np.asarray(time_s, dtype=float)
    heat_W = np.asarray(heat_W, dtype=float)

    if (
        time_s.ndim != 1
        or heat_W.shape != time_s.shape
        or len(time_s) < 2
    ):
        raise ValueError(
            "Time and heat must be matching 1D arrays."
        )

    if not (
        np.all(np.isfinite(time_s))
        and np.all(np.isfinite(heat_W))
        and np.all(np.diff(time_s) > 0)
        and np.all(heat_W >= 0)
    ):
        raise ValueError(
            "Time must increase; heat must be finite and nonnegative."
        )

    Cth = float(battery_params["thermal_capacity_J_per_K"])
    Kth = float(
        battery_params["thermal_conductance_W_per_K"]
    )

    if not (
        np.all(np.isfinite([
            Cth, Kth,
            ambient_temperature_C, initial_temperature_C
        ]))
        and Cth > 0
        and Kth >= 0
    ):
        raise ValueError("Invalid thermal parameters.")

    temperature_C = np.empty(len(time_s))
    temperature_C[0] = initial_temperature_C

    for k, dt_s in enumerate(np.diff(time_s)):
        if Kth == 0:
            # Adiabatic case: no heat exchange with ambient.
            temperature_C[k + 1] = (
                temperature_C[k] + heat_W[k] * dt_s / Cth
            )
        else:
            # expm1 improves numerical accuracy for short timesteps.
            one_minus_decay = -np.expm1(-Kth * dt_s / Cth)

            temperature_C[k + 1] = (
                temperature_C[k]
                + (
                    ambient_temperature_C
                    - temperature_C[k]
                    + heat_W[k] / Kth
                ) * one_minus_decay
            )

    return temperature_C

# ------------------------------------------------------------
# 1. APPLY TO THE EXISTING PULSE-AND-REST TEST
# ------------------------------------------------------------

battery_thermal_result = battery_rc_result.copy()

battery_thermal_result["battery_temperature_C"] = (
    simulate_battery_temperature(
        time_s=battery_thermal_result["time_s"],
        heat_W=battery_thermal_result["resistive_heat_W"],
        ambient_temperature_C=(
            environment["ambient_temperature_C"]
        ),
        initial_temperature_C=(
            battery_params["initial_temperature_C"]
        ),
    )
)

battery_thermal_result["heat_to_ambient_W"] = (
    battery_params["thermal_conductance_W_per_K"]
    * (
        battery_thermal_result["battery_temperature_C"]
        - environment["ambient_temperature_C"]
    )
)

# ------------------------------------------------------------
# 2. INDEPENDENT THERMAL CHECKS
# ------------------------------------------------------------

check_time_s = np.arange(0.0, 601.0, 1.0)
ambient_C = environment["ambient_temperature_C"]

# A battery starting at ambient with zero heat stays at ambient.
zero_heat_temperature = simulate_battery_temperature(
    check_time_s,
    np.zeros_like(check_time_s),
    ambient_C,
    ambient_C,
)

assert np.allclose(zero_heat_temperature, ambient_C)

# Constant heating must agree with the analytical solution.
test_heat_W = 100.0

constant_heat_temperature = simulate_battery_temperature(
    check_time_s,
    np.full_like(check_time_s, test_heat_W),
    ambient_C,
    ambient_C,
)

Cth = battery_params["thermal_capacity_J_per_K"]
Kth = battery_params["thermal_conductance_W_per_K"]

if Kth > 0:
    analytical_temperature = (
        ambient_C
        + test_heat_W / Kth
        * (1.0 - np.exp(-Kth * check_time_s / Cth))
    )
else:
    analytical_temperature = (
        ambient_C + test_heat_W * check_time_s / Cth
    )

assert np.allclose(
    constant_heat_temperature,
    analytical_temperature,
    atol=1e-9,
)

# A warmer battery with no heat input must cool toward ambient.
cooling_temperature = simulate_battery_temperature(
    check_time_s,
    np.zeros_like(check_time_s),
    ambient_C,
    ambient_C + 5.0,
)

assert np.all(cooling_temperature >= ambient_C - 1e-10)
assert np.all(np.diff(cooling_temperature) <= 1e-10)

# ------------------------------------------------------------
# 3. SUMMARY
# ------------------------------------------------------------

temperature_values = (
    battery_thermal_result["battery_temperature_C"].to_numpy()
)

thermal_summary = pd.DataFrame([
    (
        "Initial battery temperature",
        temperature_values[0], "°C"
    ),
    (
        "Maximum battery temperature",
        temperature_values.max(), "°C"
    ),
    (
        "Final battery temperature",
        temperature_values[-1], "°C"
    ),
    (
        "Maximum rise above initial temperature",
        temperature_values.max() - temperature_values[0], "K"
    ),
    (
        "Thermal capacity",
        Cth, "J/K"
    ),
    (
        "Thermal conductance",
        Kth, "W/K"
    ),
], columns=["Quantity", "Value", "Unit"])

display(thermal_summary.round({"Value": 5}))

# ------------------------------------------------------------
# 4. PLOTS
# ------------------------------------------------------------

thermal_figure = go.Figure()

thermal_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=battery_thermal_result["battery_temperature_C"],
    name="Model battery temperature",
))

thermal_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=np.full(len(battery_thermal_result), ambient_C),
    name="Ambient temperature",
    line=dict(dash="dash"),
))

thermal_figure.update_layout(
    title="Assumed battery thermal response | pulse-and-rest test",
    xaxis_title="Time [s]",
    yaxis_title="Temperature [°C]",
    template="plotly_white",
    height=420,
)

thermal_figure.show()

heat_figure = go.Figure()

heat_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=battery_thermal_result["resistive_heat_W"],
    name="Resistive heat generation",
))

heat_figure.add_trace(go.Scatter(
    x=battery_thermal_result["time_s"],
    y=battery_thermal_result["heat_to_ambient_W"],
    name="Heat transfer to ambient",
))

heat_figure.update_layout(
    title="Model heat generation and cooling",
    xaxis_title="Time [s]",
    yaxis_title="Heat-transfer rate [W]",
    template="plotly_white",
    height=400,
)

heat_figure.show()

print("PASS: zero heat at ambient gives constant temperature.")
print("PASS: constant heating matches the analytical solution.")
print("PASS: an initially warm battery cools toward ambient.")
print("Thermal parameters remain assumed; no physical validation.")

,Quantity,Value,Unit
0,Initial battery temperature,25.00000,°C
1,Maximum battery temperature,25.02671,°C
2,Final battery temperature,25.02618,°C
3,Maximum rise above initial temperature,0.02671,K
4,Thermal capacity,60000.00000,J/K
5,Thermal conductance,15.00000,W/K


PASS: zero heat at ambient gives constant temperature.
PASS: constant heating matches the analytical solution.
PASS: an initially warm battery cools toward ambient.
Thermal parameters remain assumed; no physical validation.


In [15]:
# STAGE 2 — CELL 9
# Interactive 3D battery simulation dashboard.
# Uses corrected electrical results and regenerates thermal results.
# No live sensor connection, BMS protection or PINN training.

required_objects = [
    "battery_rc_result",
    "battery_params",
    "simulate_battery_temperature",
    "environment",
    "build_layout",
    "widgets",
    "clear_output",
    "np", "pd", "go", "display",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run the restarted notebook through Cell 8 first. Missing: "
        + ", ".join(missing_objects)
    )

from plotly.subplots import make_subplots

# ------------------------------------------------------------
# 1. VERIFY CURRENT ELECTRICAL RESULTS
# ------------------------------------------------------------

required_columns = [
    "time_s", "pack_current_A", "SOC",
    "polarization_V", "ocv_V",
    "terminal_voltage_V", "terminal_power_W",
    "resistive_heat_W", "rc_energy_rate_W",
]

missing_columns = [
    name for name in required_columns
    if name not in battery_rc_result.columns
]

if missing_columns:
    raise RuntimeError(
        "Electrical results are incomplete. Rerun corrected Cell 7. "
        "Missing: " + ", ".join(missing_columns)
    )

electrical_data = battery_rc_result[required_columns]

if not np.all(np.isfinite(
    electrical_data.to_numpy(dtype=float)
)):
    raise ValueError("Electrical results contain invalid values.")

expected_heat_W = (
    np.square(battery_rc_result["pack_current_A"])
    * battery_params["R0_pack_ohm"]
    + np.square(battery_rc_result["polarization_V"])
    / battery_params["R1_pack_ohm"]
)

if not np.allclose(
    battery_rc_result["resistive_heat_W"],
    expected_heat_W,
    rtol=1e-10,
    atol=1e-9,
):
    raise RuntimeError(
        "Heat results do not match the corrected model. "
        "Rerun corrected Cell 7, then Cell 8."
    )

balance_error_W = (
    battery_rc_result["ocv_V"]
    * battery_rc_result["pack_current_A"]
    - battery_rc_result["terminal_power_W"]
    - battery_rc_result["resistive_heat_W"]
    - battery_rc_result["rc_energy_rate_W"]
)

if not np.all(np.abs(balance_error_W) <= 1e-9):
    raise RuntimeError(
        "Electrical power balance failed. Resolve Cell 7 first."
    )

# ------------------------------------------------------------
# 2. REGENERATE THERMAL RESULTS FROM CORRECTED HEAT
# ------------------------------------------------------------

battery_thermal_result = battery_rc_result.copy()

battery_thermal_result["battery_temperature_C"] = (
    simulate_battery_temperature(
        time_s=battery_thermal_result["time_s"],
        heat_W=battery_thermal_result["resistive_heat_W"],
        ambient_temperature_C=(
            environment["ambient_temperature_C"]
        ),
        initial_temperature_C=(
            battery_params["initial_temperature_C"]
        ),
    )
)

battery_thermal_result["heat_to_ambient_W"] = (
    battery_params["thermal_conductance_W_per_K"]
    * (
        battery_thermal_result["battery_temperature_C"]
        - environment["ambient_temperature_C"]
    )
)

# Dedicated snapshot for dashboard playback.
battery_dashboard_data = (
    battery_thermal_result.copy().reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. DASHBOARD CONTROLS
# Slider selects a stored sample; it does not change the model.
# ------------------------------------------------------------

battery_time_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(battery_dashboard_data) - 1,
    step=1,
    description="Sample:",
    continuous_update=False,
    layout=widgets.Layout(width="95%"),
)

battery_view_selector = widgets.Dropdown(
    options=["Isometric", "Side", "Front", "Top"],
    value="Isometric",
    description="View:",
)

battery_body_toggle = widgets.Checkbox(
    value=False,
    description="Show body / canopy",
    indent=False,
)

battery_dashboard_metrics = widgets.HTML()
battery_dashboard_output = widgets.Output()

def battery_metric_card(label, value):
    return (
        "<div style='padding:10px 14px;"
        "border:1px solid #d8e0e8;border-radius:8px;"
        "background:#f5f8fc;min-width:130px'>"
        f"<div style='font-size:12px;color:#506070'>{label}</div>"
        f"<div style='font-size:21px;font-weight:bold'>{value}</div>"
        "</div>"
    )

def refresh_battery_dashboard(change=None):
    sample = battery_dashboard_data.iloc[
        battery_time_slider.value
    ]

    current_A = float(sample["pack_current_A"])

    if current_A > 1e-9:
        operating_state = "Discharging"
    elif current_A < -1e-9:
        operating_state = "Charging"
    else:
        operating_state = "Rest"

    metrics = [
        ("Simulation time", f"{sample['time_s']:.0f} s"),
        ("SOC", f"{100.0 * sample['SOC']:.3f}%"),
        ("Terminal voltage",
         f"{sample['terminal_voltage_V']:.3f} V"),
        ("Pack current", f"{current_A:.1f} A"),
        ("Battery temperature",
         f"{sample['battery_temperature_C']:.4f} °C"),
        ("Terminal power",
         f"{sample['terminal_power_W']:.2f} W"),
        ("Resistive heating",
         f"{sample['resistive_heat_W']:.3f} W"),
        ("Operating state", operating_state),
    ]

    battery_dashboard_metrics.value = (
        "<div style='font-family:Arial,sans-serif'>"
        "<h3>Battery subsystem — simulated playback</h3>"
        "<div style='display:flex;flex-wrap:wrap;gap:8px'>"
        + "".join(
            battery_metric_card(label, value)
            for label, value in metrics
        )
        + "</div>"
        "<p style='font-size:12px'>"
        "Battery colour represents SOC, not temperature or safety. "
        "Parameters remain assumed. BMS protection is not yet modelled. "
        "Motor and inverter operating states are not yet modelled."
        "</p></div>"
    )

    # Reuse the existing 3D layout without changing packaging.
    vehicle_figure = build_layout(
        selected_unit="Battery",
        show_body=battery_body_toggle.value,
        show_labels=True,
        view=battery_view_selector.value,
    )

    soc_fraction = float(np.clip(sample["SOC"], 0.0, 1.0))

    # Red-to-green display mapping over the full 0–100% SOC range.
    battery_colour = (
        f"rgb({int(220 * (1.0 - soc_fraction))},"
        f"{int(80 + 140 * soc_fraction)},70)"
    )

    battery_mesh_found = False

    for trace in vehicle_figure.data:
        if trace.type == "mesh3d" and trace.name == "Battery":
            trace.color = battery_colour
            trace.opacity = 0.95
            trace.hovertemplate = (
                "<b>Battery — simulated state</b>"
                f"<br>SOC: {100 * sample['SOC']:.3f}%"
                f"<br>Voltage: {sample['terminal_voltage_V']:.3f} V"
                f"<br>Current: {current_A:.1f} A"
                f"<br>Temperature: "
                f"{sample['battery_temperature_C']:.4f} °C"
                "<br>Geometry and parameters are assumed"
                "<extra></extra>"
            )
            battery_mesh_found = True

    if not battery_mesh_found:
        raise RuntimeError(
            "Battery mesh not found. "
            "Rerun the restarted notebook's Cell 2."
        )

    vehicle_figure.update_layout(
        title=(
            "Electric tricycle | battery simulation playback"
            f"<br><sup>t = {sample['time_s']:.0f} s | "
            f"{operating_state} | assumed packaging geometry</sup>"
        ),
        height=550,
    )

    history_figure = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        subplot_titles=[
            "State of charge",
            "Terminal voltage",
            "Battery temperature",
        ],
        vertical_spacing=0.10,
    )

    histories = [
        ("SOC", 100.0, "SOC [%]"),
        ("terminal_voltage_V", 1.0, "Voltage [V]"),
        ("battery_temperature_C", 1.0, "Temperature [°C]"),
    ]

    for row_number, (column, scale, axis_label) in enumerate(
        histories, start=1
    ):
        history_figure.add_trace(
            go.Scatter(
                x=battery_dashboard_data["time_s"],
                y=battery_dashboard_data[column] * scale,
                mode="lines",
                name=axis_label,
                showlegend=False,
            ),
            row=row_number, col=1,
        )

        history_figure.add_trace(
            go.Scatter(
                x=[sample["time_s"]],
                y=[sample[column] * scale],
                mode="markers",
                marker=dict(size=9, color="#e74c3c"),
                showlegend=False,
            ),
            row=row_number, col=1,
        )

        history_figure.add_vline(
            x=float(sample["time_s"]),
            line_dash="dot",
            line_color="#e74c3c",
            row=row_number, col=1,
        )

        history_figure.update_yaxes(
            title_text=axis_label,
            row=row_number, col=1,
        )

    history_figure.update_xaxes(
        title_text="Simulation time [s]", row=3, col=1
    )

    history_figure.update_layout(
        height=650,
        template="plotly_white",
        title="Battery history | red markers show selected sample",
    )

    with battery_dashboard_output:
        clear_output(wait=True)
        vehicle_figure.show(config={"displaylogo": False})
        history_figure.show(config={"displaylogo": False})

for control in [
    battery_time_slider,
    battery_view_selector,
    battery_body_toggle,
]:
    control.observe(refresh_battery_dashboard, names="value")

display(widgets.VBox([
    widgets.HBox([
        battery_view_selector,
        battery_body_toggle,
    ]),
    battery_time_slider,
    battery_dashboard_metrics,
    battery_dashboard_output,
]))

refresh_battery_dashboard()

print("PASS: corrected heat and electrical power balance checked.")
print("Thermal results regenerated from current electrical results.")
print("Move the sample slider to inspect the simulated experiment.")

PASS: corrected heat and electrical power balance checked.
Thermal results regenerated from current electrical results.
Move the sample slider to inspect the simulated experiment.
